## v2.1 — v1.1 + feature engineering (XGBoost, HistGradientBoosting)

Same NaN-handling setup as [v1.1.ipynb](./v1.1.ipynb) (`BoostingNaNImputer`
smart-fills `acc_balance`/`prod_count`; `country`/`credit_score` are left as
raw `NaN` for native missing-value handling; both a raw-NaN and a
smart-imputed variant are tested). This notebook adds the feature engineering
from `docs/Nishkarsh/feature_engineering.md` on top, to see whether it moves
the needle over the v1.1 baseline:

```text
XGBoost  raw-NaN     : OOF F1=0.6567
XGBoost  smart-impute: OOF F1=0.6545
HistGB   raw-NaN     : OOF F1=0.6562
HistGB   smart-impute: OOF F1=0.6553
```


In [12]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42


In [13]:
train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')
print("train:", train.shape, "test:", test.shape)
print(train.isna().sum())


train: (90000, 14) test: (30000, 13)
id                     0
customer_id            0
last_name              0
credit_score        9556
country             6021
gender                 0
age                    0
tenure                 0
acc_balance         7257
prod_count          4863
has_card               0
is_active              0
estimated_salary       0
exit_status            0
dtype: int64


### Feature engineering (from `docs/Nishkarsh/feature_engineering.md`)

New columns, added on top of the imputed data, never replacing the originals:

* `balance_zero` — `acc_balance == 0` as an explicit flag. The column is
  bimodal with a huge spike at exactly zero; a flag makes that discontinuity
  explicit instead of relying on the model to discover it (NaN-aware: rows
  with an unknown balance get an unknown flag, not a fabricated 0/1).
* `country_gender` — interaction of `country` and `gender`, since churn by
  country is itself gender-skewed in this dataset.
* `active_prodcount` — interaction of `is_active` and `prod_count`, both
  independently strong churn predictors.
* `balance_salary_ratio`, `balance_age_ratio` — `acc_balance / estimated_salary`
  and `acc_balance / age`. Minor but consistent contributors in public
  notebooks on the source dataset. Division guards against a zero
  denominator (doesn't occur in this data, but cheap to guard).
* `credit_score_band` — standard FICO-style bands (Poor/Fair/Good/VeryGood/
  Excellent). Included for completeness, but `credit_score` had ~0
  correlation with everything in the original EDA, so this one is a
  low-priority, low-expectation addition.

`prod_count` itself is already treated as categorical (one-hot / native
category dtype) rather than numeric in v1.1/v1.2, since its relationship with
churn is strongly non-monotonic (2 is the *safest* count) — that's carried
over unchanged, it isn't new here.

Recall the biggest documented finding on this dataset: a 300+-pipeline sweep
on the source Kaggle competition found engineered features gave only small
gains, with model/encoding choice mattering more. So the honest goal here is
to measure the actual delta against the v1 baseline, not to assume this helps.


In [14]:
def add_engineered_features(df):
    df = df.copy()

    df['balance_zero'] = np.where(
        df['acc_balance'].isna(), np.nan, (df['acc_balance'] == 0).astype(float)
    )
    df['country_gender'] = np.where(
        df['country'].isna(), np.nan,
        df['country'].astype(str) + '_' + df['gender'].astype(str)
    )
    df['active_prodcount'] = np.where(
        df['prod_count'].isna(), np.nan,
        df['is_active'].astype(int).astype(str) + '_' + df['prod_count'].astype('Int64').astype(str)
    )
    df['balance_salary_ratio'] = df['acc_balance'] / df['estimated_salary'].replace(0, np.nan)
    df['balance_age_ratio'] = df['acc_balance'] / df['age'].replace(0, np.nan)

    credit_bins = [-np.inf, 579, 669, 739, 799, np.inf]
    credit_labels = ['Poor', 'Fair', 'Good', 'VeryGood', 'Excellent']
    df['credit_score_band'] = pd.cut(df['credit_score'], bins=credit_bins, labels=credit_labels)

    return df


### Fixed category vocabularies

Same reasoning as v1.1: XGBoost (`enable_categorical=True`) and
`HistGradientBoostingClassifier` (`categorical_features="from_dtype"`) both
need identical category sets at fit and predict time, across every CV fold,
train, and test -- otherwise the same category string can silently map to a
different internal code. Vocabularies (including the two new interaction
columns) are fixed once from the full, feature-engineered training set.


In [15]:
train_fe = add_engineered_features(train)

COUNTRY_CATS = sorted(train_fe['country'].dropna().unique().tolist())
GENDER_CATS = sorted(train_fe['gender'].dropna().unique().tolist())
PRODCOUNT_CATS = sorted(train_fe['prod_count'].dropna().unique().astype(int).tolist())
COUNTRY_GENDER_CATS = sorted(train_fe['country_gender'].dropna().unique().tolist())
ACTIVE_PRODCOUNT_CATS = sorted(train_fe['active_prodcount'].dropna().unique().tolist())
CREDIT_BAND_CATS = ['Poor', 'Fair', 'Good', 'VeryGood', 'Excellent']

ID_COLS = ['id', 'customer_id', 'last_name']
TARGET = 'exit_status'
PROD_COUNT_IMPUTE_FEATURES = [
    'age', 'is_active', 'acc_balance', 'country', 'credit_score',
    'has_card', 'estimated_salary', 'tenure',
]

print("country_gender:", COUNTRY_GENDER_CATS)
print("active_prodcount:", ACTIVE_PRODCOUNT_CATS)


country_gender: ['France_Female', 'France_Male', 'Germany_Female', 'Germany_Male', 'Spain_Female', 'Spain_Male']
active_prodcount: ['0_1', '0_2', '0_3', '0_4', '1_1', '1_2', '1_3', '1_4']


In [16]:
def cast_categoricals(df):
    """Fix dtype only -- never touches which values are missing."""
    df = df.copy()
    df['country'] = df['country'].astype(pd.CategoricalDtype(categories=COUNTRY_CATS))
    df['gender'] = df['gender'].astype(pd.CategoricalDtype(categories=GENDER_CATS))
    if 'prod_count' in df.columns:
        pc_int = df['prod_count'].astype('Int64')
        df['prod_count'] = pc_int.astype(pd.CategoricalDtype(categories=PRODCOUNT_CATS))
    if 'country_gender' in df.columns:
        df['country_gender'] = df['country_gender'].astype(pd.CategoricalDtype(categories=COUNTRY_GENDER_CATS))
    if 'active_prodcount' in df.columns:
        df['active_prodcount'] = df['active_prodcount'].astype(pd.CategoricalDtype(categories=ACTIVE_PRODCOUNT_CATS))
    if 'credit_score_band' in df.columns:
        df['credit_score_band'] = df['credit_score_band'].astype(pd.CategoricalDtype(categories=CREDIT_BAND_CATS))
    return df


def make_X(df):
    return df.drop(columns=[c for c in ID_COLS + [TARGET] if c in df.columns])


### `BoostingNaNImputer`

Unchanged from v1.1 -- fills only `acc_balance` and `prod_count`, on the
*original* columns, before feature engineering runs. `country` and
`credit_score` stay raw `NaN`.


In [17]:
class BoostingNaNImputer(BaseEstimator, TransformerMixin):
    def __init__(self, prod_count_features=PROD_COUNT_IMPUTE_FEATURES, random_state=RANDOM_STATE):
        self.prod_count_features = prod_count_features
        self.random_state = random_state

    def _country_key(self, df):
        return df['country'].astype('object').where(df['country'].notna(), '__NA__')

    def fit(self, X, y=None):
        X = X.copy()
        key = self._country_key(X)
        self.balance_median_by_country_ = X.groupby(key)['acc_balance'].median()
        self.balance_global_median_ = X['acc_balance'].median()

        known = X.dropna(subset=['prod_count'])
        Xk = known[self.prod_count_features].copy()
        Xk['country'] = Xk['country'].astype(pd.CategoricalDtype(categories=COUNTRY_CATS))
        yk_raw = known['prod_count'].astype(int)

        # xgboost's sklearn API requires contiguous zero-indexed class labels;
        # remap to positional indices and invert at predict time.
        self.prod_count_classes_ = np.sort(yk_raw.unique())
        class_to_idx = {c: i for i, c in enumerate(self.prod_count_classes_)}
        yk = yk_raw.map(class_to_idx)

        self.prod_count_model_ = XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.1,
            enable_categorical=True, tree_method='hist',
            random_state=self.random_state, verbosity=0,
        )
        self.prod_count_model_.fit(Xk, yk)
        return self

    def transform(self, X):
        X = X.copy()
        key = self._country_key(X)
        med_map = key.map(self.balance_median_by_country_)
        X['acc_balance'] = X['acc_balance'].fillna(med_map).fillna(self.balance_global_median_)

        missing = X['prod_count'].isna()
        if missing.any():
            Xm = X.loc[missing, self.prod_count_features].copy()
            Xm['country'] = Xm['country'].astype(pd.CategoricalDtype(categories=COUNTRY_CATS))
            idx_preds = self.prod_count_model_.predict(Xm)
            X.loc[missing, 'prod_count'] = self.prod_count_classes_[idx_preds]
        return X


### OOF cross-validation + F1 threshold search

Same harness as v1.1, with `add_engineered_features` inserted between the
(optional) imputer and the categorical dtype casting.


In [18]:
def oof_threshold_search(build_model_fn, X, y, use_imputer, n_splits=5, random_state=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof = np.zeros(len(X))

    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr = y.iloc[tr_idx]

        if use_imputer:
            imp = BoostingNaNImputer()
            imp.fit(X_tr)
            X_tr, X_va = imp.transform(X_tr), imp.transform(X_va)

        X_tr, X_va = add_engineered_features(X_tr), add_engineered_features(X_va)
        X_tr, X_va = cast_categoricals(X_tr), cast_categoricals(X_va)

        model = build_model_fn()
        sw = compute_sample_weight('balanced', y_tr)
        model.fit(X_tr, y_tr, sample_weight=sw)
        oof[va_idx] = model.predict_proba(X_va)[:, 1]

    thresholds = np.linspace(0.02, 0.98, 97)
    f1s = [f1_score(y, oof > t) for t in thresholds]
    best = int(np.argmax(f1s))
    return oof, thresholds[best], f1s[best]


In [19]:
X = make_X(train)
y = train[TARGET]

def build_xgb():
    return XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        enable_categorical=True, tree_method='hist',
        random_state=RANDOM_STATE, verbosity=0,
    )

_, thr_xgb_raw, f1_xgb_raw = oof_threshold_search(build_xgb, X, y, use_imputer=False)
_, thr_xgb_imp, f1_xgb_imp = oof_threshold_search(build_xgb, X, y, use_imputer=True)

print(f"XGBoost  raw-NaN     : OOF F1={f1_xgb_raw:.4f}  best_threshold={thr_xgb_raw:.2f}")
print(f"XGBoost  smart-impute: OOF F1={f1_xgb_imp:.4f}  best_threshold={thr_xgb_imp:.2f}")


XGBoost  raw-NaN     : OOF F1=0.6556  best_threshold=0.63
XGBoost  smart-impute: OOF F1=0.6541  best_threshold=0.62


In [20]:
def build_hgb():
    return HistGradientBoostingClassifier(
        max_depth=6, learning_rate=0.05, max_iter=400,
        categorical_features='from_dtype',
        random_state=RANDOM_STATE,
    )

_, thr_hgb_raw, f1_hgb_raw = oof_threshold_search(build_hgb, X, y, use_imputer=False)
_, thr_hgb_imp, f1_hgb_imp = oof_threshold_search(build_hgb, X, y, use_imputer=True)

print(f"HistGB   raw-NaN     : OOF F1={f1_hgb_raw:.4f}  best_threshold={thr_hgb_raw:.2f}")
print(f"HistGB   smart-impute: OOF F1={f1_hgb_imp:.4f}  best_threshold={thr_hgb_imp:.2f}")


HistGB   raw-NaN     : OOF F1=0.6554  best_threshold=0.62
HistGB   smart-impute: OOF F1=0.6543  best_threshold=0.62


In [21]:
results = pd.DataFrame([
    {'model': 'XGBoost', 'variant': 'raw-NaN', 'oof_f1': f1_xgb_raw, 'threshold': thr_xgb_raw},
    {'model': 'XGBoost', 'variant': 'smart-impute', 'oof_f1': f1_xgb_imp, 'threshold': thr_xgb_imp},
    {'model': 'HistGB', 'variant': 'raw-NaN', 'oof_f1': f1_hgb_raw, 'threshold': thr_hgb_raw},
    {'model': 'HistGB', 'variant': 'smart-impute', 'oof_f1': f1_hgb_imp, 'threshold': thr_hgb_imp},
]).sort_values('oof_f1', ascending=False).reset_index(drop=True)

v1_1_baseline = pd.DataFrame([
    {'model': 'XGBoost', 'variant': 'raw-NaN', 'v1_1_oof_f1': 0.6567},
    {'model': 'XGBoost', 'variant': 'smart-impute', 'v1_1_oof_f1': 0.6545},
    {'model': 'HistGB', 'variant': 'raw-NaN', 'v1_1_oof_f1': 0.6562},
    {'model': 'HistGB', 'variant': 'smart-impute', 'v1_1_oof_f1': 0.6553},
])
comparison = results.merge(v1_1_baseline, on=['model', 'variant'])
comparison['delta_vs_v1_1'] = comparison['oof_f1'] - comparison['v1_1_oof_f1']
comparison


,model,variant,oof_f1,threshold,v1_1_oof_f1,delta_vs_v1_1
0,XGBoost,raw-NaN,0.655578,0.63,0.6567,-0.001122
1,HistGB,raw-NaN,0.655441,0.62,0.6562,-0.000759
2,HistGB,smart-impute,0.654333,0.62,0.6553,-0.000967
3,XGBoost,smart-impute,0.654081,0.62,0.6545,-0.000419


### Final fit + submission

Refits whichever `(model, variant)` combination scored best above on the
full training set and writes a submission.


In [ ]:
import os

BUILDERS = {'XGBoost': build_xgb, 'HistGB': build_hgb}
best_row = results.iloc[0]
best_model_name = best_row['model']
best_use_imputer = best_row['variant'] == 'smart-impute'
best_threshold = best_row['threshold']
print(f"Refitting best combo: {best_model_name} / {best_row['variant']} (OOF F1={best_row['oof_f1']:.4f})")

X_train_final = make_X(train)
y_train_final = train[TARGET]
X_test_final = make_X(test)

if best_use_imputer:
    final_imputer = BoostingNaNImputer()
    final_imputer.fit(X_train_final)
    X_train_final = final_imputer.transform(X_train_final)
    X_test_final = final_imputer.transform(X_test_final)

X_train_final = cast_categoricals(add_engineered_features(X_train_final))
X_test_final = cast_categoricals(add_engineered_features(X_test_final))

final_model = BUILDERS[best_model_name]()
sw_final = compute_sample_weight('balanced', y_train_final)
final_model.fit(X_train_final, y_train_final, sample_weight=sw_final)

test_proba = final_model.predict_proba(X_test_final)[:, 1]
test_pred = (test_proba > best_threshold).astype(int)

os.makedirs('outputs', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'exit_status': test_pred})
submission.to_csv(f'outputs/v2_1_{best_model_name.lower()}_submission.csv', index=False)
submission.head()


Refitting best combo: XGBoost / raw-NaN (OOF F1=0.6556)


,id,exit_status
0,0,0
1,1,1
2,2,1
3,3,0
4,4,0


: 